# 3. Trips and the Benchmark

Every trip time on the map was worked out ahead of time and saved in the travel-time file. This notebook checks the trips the story leans on:

1. **The 18.8 million trip calculations** the Methodology describes.
2. **The county's typical trips**: a 7.3-minute drive and a 39-minute bus ride to the fastest-to-reach supermarket.
3. **One mile is not one mile**: the Kroger that sits under a mile away in a straight line but takes 1.7 miles of road to reach.
4. **The Northridge grocery run**: 22 departure times on the 17 bus, every bus checked against GDRTA's own timetable.
5. **The Google Maps benchmark**: how closely our trip times match Google's on 52 trips and 60 walks.

Sections 3 to 5 read route traces from `engine_witness.json`, written by the site's own routing engine (`src/mobility.js`), and verify them directly against GDRTA's timetable and Google Maps.

**How to read this notebook:** Each grey box runs a small piece of Python and prints what it found right below it. A box that starts with `assert` is a check: if any number disagrees with what the site publishes, the notebook stops with an error. Each check is written out in words just above it.

In [1]:
import json
import math
import statistics
from collections import Counter
from pathlib import Path

snapshots_folder = Path('..') / 'snapshots'


def count_of(items):
    return len(items)


def rounded_to_tenth(number):
    return round(number, 1)

travel_surface = json.loads((snapshots_folder / 'travel-surface@2026-10-03.json').read_text())
engine_witness = json.loads(Path('engine_witness.json').read_text())
hexagons = travel_surface['cells']
lived_in_hexagons = [hexagon for hexagon in hexagons if hexagon['populated']]


def drive_minutes_to_supermarket(hexagon):
    return hexagon['layers']['supermarket']['car']['minutes']


def bus_trip_to_supermarket(hexagon, period='peak'):
    return hexagon['layers']['supermarket']['transit'][period]

## 1. The 18.8 million trip calculations

For each of the 8,568 hexagons, the routing engine times trips to all 548 food places across four ways of traveling (driving, walking, the morning rush-hour bus and the midday bus): a total of **18.8 million** trip calculations.

In [2]:
ways_of_traveling = ['car', 'walk', 'transit at rush hour', 'transit at midday']
supermarket_layer = hexagons[0]['layers']['supermarket']
recorded_ways = ['car', 'walk']
for period in supermarket_layer['transit']:
    recorded_ways.append('transit at rush hour' if period == 'peak' else 'transit at midday')

food_places = 548
trip_calculations = count_of(hexagons) * food_places * count_of(ways_of_traveling)

print(f"ways of traveling recorded for each hexagon: {', '.join(recorded_ways)}")
print(f'{count_of(hexagons):,} hexagons × {food_places} food places × {count_of(ways_of_traveling)} ways')
print(f'= {trip_calculations:,} trip calculations, about {trip_calculations / 1e6:.1f} million')

ways of traveling recorded for each hexagon: car, walk, transit at rush hour, transit at midday
8,568 hexagons × 548 food places × 4 ways
= 18,781,056 trip calculations, about 18.8 million


**Check:** the four ways of traveling, 8,568 hexagons, and about 18.8 million trip calculations.

In [3]:
assert recorded_ways == ways_of_traveling
assert count_of(hexagons) == 8_568
assert rounded_to_tenth(trip_calculations / 1e6) == 18.8

## 2. The county's typical trips

**Driving.** Across the 2,933 lived-in hexagons, the median drive to the fastest-to-reach supermarket is 7.3 minutes, and almost every lived-in hexagon is within 15 minutes of one.

In [4]:
drive_minutes = sorted(drive_minutes_to_supermarket(hexagon) for hexagon in lived_in_hexagons)
median_drive_minutes = statistics.median(drive_minutes)
drives_of_15_minutes_or_more = count_of([minutes for minutes in drive_minutes if round(minutes) >= 15])

print(f'median drive: {median_drive_minutes:.2f} minutes')
print(f'lived-in hexagons 15 minutes or more from a supermarket by car: {drives_of_15_minutes_or_more} of {count_of(drive_minutes):,}')
print(f'the longest drive is {drive_minutes[-1]:.1f} minutes')

median drive: 7.27 minutes
lived-in hexagons 15 minutes or more from a supermarket by car: 45 of 2,933
the longest drive is 20.0 minutes


**Check:** the median drive rounds to 7.3 minutes, and fewer than 2 in every 100 lived-in hexagons are 15 minutes or more from a supermarket by car.

In [5]:
assert rounded_to_tenth(median_drive_minutes) == 7.3
assert drives_of_15_minutes_or_more / count_of(drive_minutes) < 0.02

**By bus.** Across all 2,933 lived-in hexagons, the median rush-hour bus trip to a supermarket is **39 minutes**. That includes **921 lived-in hexagons with no bus service at all** (represented in code as `math.inf` so they sort after every reachable trip). Among the **2,012 hexagons where a bus does run**, the median trip is **30 minutes**.

In [6]:
bus_minutes_for_all_hexagons = []
bus_minutes_where_a_bus_runs = []
for hexagon in lived_in_hexagons:
    trip = bus_trip_to_supermarket(hexagon)
    if trip is None:
        bus_minutes_for_all_hexagons.append(math.inf)
    else:
        bus_minutes_for_all_hexagons.append(trip['minutes'])
        bus_minutes_where_a_bus_runs.append(trip['minutes'])
hexagons_with_no_bus_trip = count_of(lived_in_hexagons) - count_of(bus_minutes_where_a_bus_runs)

median_bus_minutes = statistics.median(bus_minutes_for_all_hexagons)
median_bus_minutes_where_a_bus_runs = statistics.median(bus_minutes_where_a_bus_runs)

print(f'lived-in hexagons with no bus trip at rush hour: {hexagons_with_no_bus_trip}')
print(f'median bus trip, counting those as longest: {median_bus_minutes:.2f} minutes')
print(f'median bus trip, where a bus runs:          {median_bus_minutes_where_a_bus_runs:.2f} minutes')

lived-in hexagons with no bus trip at rush hour: 921
median bus trip, counting those as longest: 38.75 minutes
median bus trip, where a bus runs:          30.32 minutes


**Check:** the median bus trip rounds to 39 minutes, and 921 lived-in hexagons have no bus trip at rush hour.

In [7]:
assert round(median_bus_minutes) == 39
assert hexagons_with_no_bus_trip == 921

## 3. One mile is not one mile

The USDA measures urban food access using a 1-mile straight-line circle. Story steps 5 and 6 show why straight lines can mislead: a home near Kroger #770 sits **0.79 miles** away as the crow flies, but walking along real streets takes **1.71 miles**. The straight line is computed below; the walk comes from the routing engine.

In [8]:
earth_radius_mi = 3958.8


def straight_line_miles(start, end):
    start_lat, end_lat = math.radians(start['lat']), math.radians(end['lat'])
    lat_change, lon_change = end_lat - start_lat, math.radians(end['lon'] - start['lon'])
    haversine = math.sin(lat_change / 2) ** 2 + math.cos(start_lat) * math.cos(end_lat) * math.sin(lon_change / 2) ** 2
    return 2 * earth_radius_mi * math.asin(math.sqrt(haversine))


kroger_trip = engine_witness['straightLine']
kroger_straight_miles = straight_line_miles(kroger_trip['from'], kroger_trip['store'])

print(f"{kroger_trip['store']['name']}: {kroger_straight_miles:.2f} miles in a straight line")
print(f"walking on the streets: {kroger_trip['walkMiles']:.2f} miles ({kroger_trip['walkMiles'] / kroger_straight_miles:.1f} times as far)")
print(f"driving on the streets: {kroger_trip['driveMiles']:.2f} miles")

Kroger #770: 0.79 miles in a straight line
walking on the streets: 1.71 miles (2.2 times as far)
driving on the streets: 1.78 miles


**Check:** under a mile in a straight line, and between 1.5 and 2 miles on foot.

In [9]:
assert kroger_straight_miles < 1
assert 1.5 < kroger_trip['walkMiles'] < 2

## 4. The Northridge grocery run

In story steps 10 to 19, we follow a round-trip grocery errand from Northridge to the Walmart Supercenter on Route 17 across **22 departure times** (every 10 minutes from 10:00 AM to 1:30 PM), comparing a 40-minute shop against a 50-minute shop.

In [10]:
grocery_run = engine_witness['groceryRun']


def clock(minutes_after_midnight):
    hours, minutes = divmod(round(minutes_after_midnight), 60)
    return f"{(hours - 1) % 12 + 1}:{minutes:02d} {'AM' if hours < 12 else 'PM'}"


departures = list(range(grocery_run['departures']['firstMin'], grocery_run['departures']['lastMin'] + 1, grocery_run['departures']['everyMin']))

print(grocery_run['store']['name'])
print(f"{count_of(departures)} departures from {clock(departures[0])} to {clock(departures[-1])}")
print(f"shopping {' or '.join(str(minutes) for minutes in grocery_run['shopMins'])} minutes")

Wal-Mart Supercenter #3783
22 departures from 10:00 AM to 1:30 PM
shopping 40 or 50 minutes


Here is one trip there, leaving home at 12:10 PM, step by step:

In [11]:
legs_there = {clock(leg['leaveAt']): leg for leg in grocery_run['legs'] if leg['direction'] == 'there'}

trip_there = legs_there['12:10 PM']
clock_minutes = trip_there['leaveAt']
for step in trip_there['components']:
    place = step.get('stopName') or f"{step.get('fromStop')} to {step.get('toStop')}"
    print(f"{clock(clock_minutes):>8}  {step['kind']:<20} {step['minutes']:5.1f} min  {step.get('route') or '':<9} {place}")
    clock_minutes += step['minutes']
print(f'{clock(clock_minutes):>8}  at the store')

12:10 PM  walk-to-stop           4.9 min  Route 17  Dixie @ Division
12:15 PM  wait                   3.7 min  Route 17  Dixie @ Division
12:19 PM  ride                   4.4 min  Route 17  Dixie @ Division to Walmart @ York Commons
12:23 PM  walk-to-destination    1.8 min            Walmart @ York Commons
12:25 PM  at the store


### Checking every bus ride against GDRTA's timetable

Next, we verify all 72 bus rides across those 66 trips against `GDRTA_timetable@2026-10-03.json`: every ride must match a real scheduled bus on that route at the exact boarding and arrival minutes. (Two times count as the same when they agree to a millionth of a minute.)

In [12]:
bus_timetable = json.loads((snapshots_folder / 'GDRTA_timetable@2026-10-03.json').read_text())
stops_in_timetable_order = bus_timetable['stops']


def same_time(first_minutes, second_minutes):
    return abs(first_minutes - second_minutes) < 1e-6


def scheduled_trip_for(ride):
    for pattern in bus_timetable['patterns']:
        if pattern['routeName'] != ride['route']:
            continue
        names_along_route = [stops_in_timetable_order[stop_index]['name'] for stop_index in pattern['stops']]
        if ride['fromStop'] not in names_along_route or ride['toStop'] not in names_along_route:
            continue
        boarding_position = names_along_route.index(ride['fromStop'])
        leaving_position = names_along_route.index(ride['toStop'], boarding_position + 1)
        for trip_times in pattern['tripTimes']:
            if same_time(trip_times[boarding_position], ride['departsAt']) and same_time(trip_times[leaving_position], ride['arrivesAt']):
                return pattern, trip_times
    return None


rides = []
for leg in grocery_run['legs']:
    for step in leg['components']:
        if step['kind'] == 'ride':
            rides.append(step)
rides_on_the_timetable = [ride for ride in rides if scheduled_trip_for(ride) is not None]

print(f'{count_of(rides)} rides in the 66 trips; {count_of(rides_on_the_timetable)} match a scheduled bus to the minute')
for route, ride_count in sorted(Counter(ride['route'] for ride in rides).items()):
    print(f'{route}: {ride_count} rides')

72 rides in the 66 trips; 72 match a scheduled bus to the minute
Route 17: 66 rides
Route 22: 6 rides


We also check that every walk is timed at a steady 3 mph and every wait ends the moment the bus departs:

In [13]:
walking_mph = 3


def steps_add_up(leg):
    clock_minutes = leg['leaveAt']
    for step in leg['components']:
        if step['kind'].startswith('walk') and not same_time(step['miles'] / walking_mph * 60, step['minutes']):
            return False
        if step['kind'] == 'wait' and not same_time(clock_minutes + step['minutes'], step['busLeavesAt']):
            return False
        if step['kind'] == 'ride' and not same_time(clock_minutes, step['departsAt']):
            return False
        clock_minutes += step['minutes']
    return same_time(clock_minutes - leg['leaveAt'], leg['totalMin'])


legs_that_add_up = [leg for leg in grocery_run['legs'] if steps_add_up(leg)]

print(f"{count_of(legs_that_add_up)} of {count_of(grocery_run['legs'])} trips add up step by step")

66 of 66 trips add up step by step


**Check:** 22 departures; every one of the rides matches a scheduled bus; and every trip adds up step by step.

In [14]:
assert count_of(departures) == 22
assert count_of(rides_on_the_timetable) == count_of(rides)
assert count_of(legs_that_add_up) == count_of(grocery_run['legs'])

### The whole errand

Now put each errand together: the trip there, the shopping, and the trip home that starts the moment shopping ends. The total is the time from leaving home to getting back.

In [15]:
legs_home = [leg for leg in grocery_run['legs'] if leg['direction'] == 'back']


def trip_home_leaving_at(minutes_after_midnight):
    return next(leg for leg in legs_home if same_time(leg['leaveAt'], minutes_after_midnight))


def errand_leaving_at(leave_home_at, shopping_minutes):
    at_store = leave_home_at + legs_there[clock(leave_home_at)]['totalMin']
    shopping_ends = at_store + shopping_minutes
    trip_home = trip_home_leaving_at(shopping_ends)
    return {
        'leave_home_at': leave_home_at,
        'shopping_minutes': shopping_minutes,
        'shopping_ends': shopping_ends,
        'trip_home': trip_home,
        'total_minutes': shopping_ends + trip_home['totalMin'] - leave_home_at,
    }


errands = {
    (clock(leave_home_at), shopping_minutes): errand_leaving_at(leave_home_at, shopping_minutes)
    for leave_home_at in departures
    for shopping_minutes in grocery_run['shopMins']
}


def hours_and_minutes(minutes):
    hours, minutes_left = divmod(round(minutes), 60)
    return f'{hours} hr {minutes_left:02d} min' if hours else f'{minutes_left} min'


print('leave home   40 min shopping   50 min shopping   extra time for 10 more minutes in the store')
for leave_home_at in departures:
    shorter, longer = errands[(clock(leave_home_at), 40)], errands[(clock(leave_home_at), 50)]
    shorter_total, longer_total = shorter['total_minutes'], longer['total_minutes']
    extra = round(longer_total - shorter_total)
    print(f"{clock(leave_home_at):>10}   {hours_and_minutes(shorter_total):>15}   {hours_and_minutes(longer_total):>15}   +{extra} min")

leave home   40 min shopping   50 min shopping   extra time for 10 more minutes in the store
  10:00 AM       1 hr 28 min       2 hr 24 min   +56 min
  10:10 AM       1 hr 18 min       2 hr 14 min   +56 min
  10:20 AM       2 hr 04 min       3 hr 02 min   +58 min
  10:30 AM       1 hr 54 min       2 hr 52 min   +58 min
  10:40 AM       1 hr 44 min       2 hr 42 min   +58 min
  10:50 AM       1 hr 34 min       2 hr 32 min   +58 min
  11:00 AM       1 hr 24 min       2 hr 22 min   +58 min
  11:10 AM       1 hr 14 min       2 hr 12 min   +58 min
  11:20 AM       2 hr 02 min       3 hr 01 min   +59 min
  11:30 AM       1 hr 52 min       2 hr 51 min   +59 min
  11:40 AM       1 hr 42 min       2 hr 41 min   +59 min
  11:50 AM       1 hr 32 min       2 hr 31 min   +59 min
  12:00 PM       1 hr 22 min       2 hr 21 min   +59 min
  12:10 PM       1 hr 12 min       2 hr 11 min   +59 min
  12:20 PM       2 hr 01 min       2 hr 58 min   +57 min
  12:30 PM       1 hr 51 min       2 hr 48 min   +57

The story's claims, one by one (steps 11 to 19):

- leaving at **12:10 PM**, the 40-minute errand takes **1 hour 12 minutes**;
- leaving at **10:20 AM**, the same errand takes **2 hours 4 minutes**;
- at 12:10 PM, 40 minutes of shopping puts the rider at the stop home about **5 minutes before the 1:12 PM bus**; with 50 minutes, about **5 minutes after it**, and the next 17 leaves at **2:12 PM**;
- at every one of the 22 departures, 10 more minutes in the store adds **almost an hour**;
- by car the 40-minute errand takes about **59 minutes**, on foot about **1 hour 56 minutes**.

In [16]:
def arrival_at_stop_home(errand):
    walk_to_stop = errand['trip_home']['components'][0]
    return errand['shopping_ends'] + walk_to_stop['minutes']


def bus_home_departure(errand):
    return next(step['busLeavesAt'] for step in errand['trip_home']['components'] if step['kind'] == 'wait')


best_errand = errands[('12:10 PM', 40)]
worst_errand = errands[('10:20 AM', 40)]
longer_shop = errands[('12:10 PM', 50)]
extra_minutes = []
for leave_home_at in departures:
    extra_minutes.append(errands[(clock(leave_home_at), 50)]['total_minutes'] - errands[(clock(leave_home_at), 40)]['total_minutes'])
round_trip_by_car = 2 * grocery_run['driveOneWayMin'] + 40
round_trip_on_foot = 2 * grocery_run['walkOneWayMin'] + 40

print(f"12:10 PM, 40 minutes: {hours_and_minutes(best_errand['total_minutes'])}")
print(f"10:20 AM, 40 minutes: {hours_and_minutes(worst_errand['total_minutes'])}")
print(f"12:10 PM, 40 minutes: at the stop at {clock(arrival_at_stop_home(best_errand))}, the bus leaves at {clock(bus_home_departure(best_errand))}")
print(f"12:10 PM, 50 minutes: at the stop at {clock(arrival_at_stop_home(longer_shop))}, the next bus leaves at {clock(bus_home_departure(longer_shop))}")
print(f'10 more minutes in the store adds {round(min(extra_minutes))} to {round(max(extra_minutes))} minutes')
print(f'by car: {hours_and_minutes(round_trip_by_car)}; on foot: {hours_and_minutes(round_trip_on_foot)}')

12:10 PM, 40 minutes: 1 hr 12 min
10:20 AM, 40 minutes: 2 hr 04 min
12:10 PM, 40 minutes: at the stop at 1:07 PM, the bus leaves at 1:12 PM
12:10 PM, 50 minutes: at the stop at 1:17 PM, the next bus leaves at 2:12 PM
10 more minutes in the store adds 54 to 59 minutes
by car: 59 min; on foot: 1 hr 56 min


**Check:** 72 minutes at 12:10 PM, 124 minutes at 10:20 AM, a 5-minute margin on either side of the 1:12 PM bus, at least 50 extra minutes when shopping 10 minutes longer, 59 minutes by car, and 116 minutes on foot.

In [17]:
assert round(best_errand['total_minutes']) == 72
assert round(worst_errand['total_minutes']) == 124
assert clock(bus_home_departure(best_errand)) == '1:12 PM'
assert round(bus_home_departure(best_errand) - arrival_at_stop_home(best_errand)) == 5
assert round(arrival_at_stop_home(longer_shop) - bus_home_departure(best_errand)) == 5
assert clock(bus_home_departure(longer_shop)) == '2:12 PM'
assert min(extra_minutes) >= 50
assert round(round_trip_by_car) == 59
assert round(round_trip_on_foot) == 116

## 5. The Google Maps benchmark

To benchmark our routing engine against a commercial trip planner, we compare **50 trips** leaving at 8:00 AM on Tuesday, October 13, 2026 (plus 2 diagnostic test cases, left out of the medians) and **60 walks** against Google Maps. Because Google Maps does not include time to park, we compare its driving times against our drive times before the parking allowance.

In [18]:
google_sample = json.loads(Path('calibration_sample.json').read_text())
google_walks = json.loads(Path('calibration_walks.json').read_text())['walks']
google_by_trip = {trip['id']: trip['google'] for trip in google_sample['trips']}

engine_trips = engine_witness['calibration']['trips']
benchmark_trips = [trip for trip in engine_trips if trip['stratum'] != 'diagnostic']

diagnostic_trips = count_of(engine_trips) - count_of(benchmark_trips)
print(f'{count_of(engine_trips)} trips ({count_of(benchmark_trips)} in the benchmark, {diagnostic_trips} diagnostic), {count_of(google_walks)} walks')
for kind_of_place, trip_count in Counter(trip['stratum'] for trip in engine_trips).items():
    print(f'  {kind_of_place}: {trip_count} trips')

52 trips (50 in the benchmark, 2 diagnostic), 60 walks
  food-zone: 15 trips
  high-need: 15 trips
  urban: 7 trips
  suburban: 7 trips
  rural: 6 trips
  diagnostic: 2 trips


In [19]:
def median_gap(pairs):
    return statistics.median(abs(ours - google) for ours, google in pairs if ours is not None and google is not None)


def google_bus_arrival(trip):
    return (google_by_trip[trip['id']].get('transit') or {}).get('arrivalMin')


def google_drive_minutes(trip):
    return (google_by_trip[trip['id']].get('driveNoTraffic') or {}).get('durationMin')


bus_arrival_pairs = [(trip['arrivalMin'], google_bus_arrival(trip)) for trip in benchmark_trips]
drive_pairs = [(trip['driveMin'], google_drive_minutes(trip)) for trip in benchmark_trips]

bus_arrival_median_gap = median_gap(bus_arrival_pairs)
drive_median_gap = median_gap(drive_pairs)

print(f'bus arrival: half the trips within {bus_arrival_median_gap:.2f} minutes of Google')
print(f'driving:     half the trips within {drive_median_gap:.2f} minutes ({drive_median_gap * 60:.0f} seconds) of Google')

bus arrival: half the trips within 1.44 minutes of Google
driving:     half the trips within 0.79 minutes (47 seconds) of Google


For walking, the benchmark compares distance rather than time, because Google walks a little more slowly than our 3 miles an hour.

In [20]:
engine_walk_miles = {walk['id']: walk['miles'] for walk in engine_witness['calibration']['walks']}
feet_per_mile = 5280

walk_median_gap_miles = statistics.median(abs(engine_walk_miles[walk['id']] - walk['googleMi']) for walk in google_walks)
google_walking_mph = statistics.median(walk['googleMi'] / walk['googleMin'] * 60 for walk in google_walks)

print(f'walking: half the walks within {walk_median_gap_miles:.3f} miles ({walk_median_gap_miles * feet_per_mile:.0f} feet) of Google')
print(f"Google's walking pace: {google_walking_mph:.2f} mph, against our {walking_mph}")

walking: half the walks within 0.029 miles (154 feet) of Google
Google's walking pace: 2.71 mph, against our 3


**Check:** 52 trips and 60 walks; bus arrivals within 1.4 minutes of Google, driving within 47 seconds, walks within 154 feet, and Google walking at 2.7 mph.

In [21]:
assert count_of(engine_trips) == 52
assert count_of(google_walks) == 60
assert rounded_to_tenth(bus_arrival_median_gap) == 1.4
assert round(drive_median_gap * 60) == 47
assert round(walk_median_gap_miles * feet_per_mile) == 154
assert rounded_to_tenth(google_walking_mph) == 2.7

## Summary

| Check | Result | Where it appears on the site |
| :--- | :--- | :--- |
| Trip calculations | **8,568 × 548 × 4 ≈ 18.8 million** | Methodology §2 |
| Median drive to a supermarket, lived-in hexagons | **7.3 minutes** | Story step 9, Methodology §2 |
| Median bus trip to a supermarket at rush hour | **39 minutes**, counting the 921 hexagons with no bus trip as longest; 30 minutes where a bus runs | Story step 21 |
| Kroger: straight line against the streets | **0.8 miles** in a straight line, **1.7 miles** on foot | Story steps 5 and 6 |
| Northridge grocery run | Every bus matches GDRTA's timetable; 1 hr 12 min at 12:10 PM, 2 hr 4 min at 10:20 AM, and 10 more minutes in the store adds almost an hour at every departure | Story steps 10 to 19 |
| Google Maps benchmark | Bus arrival within **1.4 minutes**, driving within **47 seconds**, walks within **154 feet** | Methodology §2 |

The cell below outputs these key figures as JSON, so they can be checked against the numbers the live site shows.

In [22]:
audited_figures = {
    'tripCalculationsMillions': round(trip_calculations / 1e6, 1),
    'medianDriveMinutes': round(median_drive_minutes, 1),
    'medianBusMinutes': round(median_bus_minutes),
    'groceryRunDepartures': count_of(departures),
    'busArrivalMedianGapMinutes': round(bus_arrival_median_gap, 2),
    'driveMedianGapMinutes': round(drive_median_gap, 2),
    'walkMedianGapMiles': round(walk_median_gap_miles, 4),
    'googleWalkingMph': round(google_walking_mph, 2),
}

print(json.dumps(audited_figures, sort_keys=True))

{"busArrivalMedianGapMinutes": 1.44, "driveMedianGapMinutes": 0.79, "googleWalkingMph": 2.71, "groceryRunDepartures": 22, "medianBusMinutes": 39, "medianDriveMinutes": 7.3, "tripCalculationsMillions": 18.8, "walkMedianGapMiles": 0.0292}


To rerun every audit notebook without Jupyter: `python3 notebooks/run_notebooks.py` from the project folder. It needs only Python 3.10 or newer.